# Dataset Downloader & Sampler
Downloads CityFlowV2 + WILDTRACK, picks a small multi-camera sample from each,
and packages them for local testing.

**Output**: `citywide_sample.zip` (~500 MB–1 GB) with:
- CityFlowV2: 6 cameras (2 scenes) with vehicles
- WILDTRACK: all 7 cameras with people
- Ground truth annotations for both

In [ ]:
import os, sys, subprocess, shutil, zipfile
from pathlib import Path

REPO_URL = "https://github.com/MRKDaGods/gp.git"
WORK = Path("/kaggle/working")
TMP = Path("/tmp/datasets")
TMP.mkdir(parents=True, exist_ok=True)

# Show space
for m in ["/kaggle/working", "/tmp"]:
    t, u, f = shutil.disk_usage(m)
    print(f"{m}: {f/(1024**3):.1f} GB free")

In [ ]:
# Clone repo (needed for download_datasets.py)
PROJECT = WORK / "gp"
if not PROJECT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT)], check=True)
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

# Symlink data/raw to /tmp so downloads go to big disk
raw = PROJECT / "data" / "raw"
raw.parent.mkdir(parents=True, exist_ok=True)
if raw.exists() and not raw.is_symlink():
    shutil.rmtree(raw)
if not raw.exists():
    raw.symlink_to(TMP)
print(f"data/raw -> {TMP}")

In [ ]:
!pip install -q gdown opencv-python-headless click 2>&1 | tail -3

## 1. Download WILDTRACK (~6.3 GB)

In [ ]:
!python scripts/download_datasets.py --dataset wildtrack

In [ ]:
# Verify WILDTRACK
wt = TMP / "wildtrack"
if wt.exists():
    for d in ["videos", "Image_subsets", "annotations_positions"]:
        p = wt / d
        if p.exists():
            items = list(p.iterdir())
            print(f"  {d}: {len(items)} items")
else:
    print("WILDTRACK not found!")

## 2. Download CityFlowV2 (~15–20 GB)

In [ ]:
!python scripts/download_datasets.py --dataset cityflowv2

In [ ]:
# Verify CityFlowV2 — list all cameras
cfv2 = TMP / "cityflowv2"
if cfv2.exists():
    cameras = sorted([d.name for d in cfv2.iterdir() if d.is_dir()])
    print(f"CityFlowV2: {len(cameras)} cameras")
    for cam in cameras:
        cam_dir = cfv2 / cam
        vids = list(cam_dir.glob("*.avi")) + list(cam_dir.glob("*.mp4"))
        gt = cam_dir / "gt.txt"
        gt_str = "+ GT" if gt.exists() else "no GT"
        vid_size = sum(v.stat().st_size for v in vids) / (1024**2)
        print(f"  {cam}: {len(vids)} video(s) ({vid_size:.0f} MB) {gt_str}")
else:
    print("CityFlowV2 not found!")

## 3. Build Sample Package

Pick representative cameras and package for local download:
- **CityFlowV2**: 2 scenes x 3 cameras = 6 cameras (vehicles at intersections)
- **WILDTRACK**: all 7 cameras (people at 1080p, as MP4 video + annotations)

In [ ]:
import re

SAMPLE_DIR = Path("/tmp/sample")
if SAMPLE_DIR.exists():
    shutil.rmtree(SAMPLE_DIR)

# --- CityFlowV2: pick first 3 cameras from 2 different scenes ---
cfv2_sample = SAMPLE_DIR / "cityflowv2"
if cfv2.exists():
    cameras = sorted([d for d in cfv2.iterdir() if d.is_dir()])
    # Group by scene
    scenes = {}
    for cam in cameras:
        scene = cam.name.split("_")[0]  # e.g. S01
        scenes.setdefault(scene, []).append(cam)

    # Pick first 2 scenes, 3 cameras each
    selected = []
    for scene_name in sorted(scenes.keys())[:2]:
        selected.extend(scenes[scene_name][:3])

    print(f"CityFlowV2 sample: {len(selected)} cameras")
    for cam_dir in selected:
        dest = cfv2_sample / cam_dir.name
        dest.mkdir(parents=True, exist_ok=True)
        # Copy video + gt
        for f in cam_dir.iterdir():
            if f.is_file():
                shutil.copy2(f, dest / f.name)
        size = sum(f.stat().st_size for f in dest.iterdir()) / (1024**2)
        print(f"  {cam_dir.name}: {size:.0f} MB")

# --- WILDTRACK: all 7 cameras (video + annotations) ---
wt_sample = SAMPLE_DIR / "wildtrack"
if wt.exists():
    # Copy videos
    vid_dir = wt / "videos"
    if vid_dir.exists():
        dest_vid = wt_sample / "videos"
        dest_vid.mkdir(parents=True, exist_ok=True)
        for v in sorted(vid_dir.glob("*.mp4")):
            shutil.copy2(v, dest_vid / v.name)
            print(f"  {v.name}: {v.stat().st_size/(1024**2):.0f} MB")

    # Copy annotations
    ann_dir = wt / "annotations_positions"
    if ann_dir.exists():
        dest_ann = wt_sample / "annotations_positions"
        shutil.copytree(ann_dir, dest_ann)
        n = len(list(dest_ann.glob("*.json")))
        print(f"  annotations: {n} JSON files")

# Total size
total = sum(f.stat().st_size for f in SAMPLE_DIR.rglob("*") if f.is_file())
print(f"\nTotal sample size: {total/(1024**3):.2f} GB")

In [ ]:
# Package as zip for download
ZIP_PATH = WORK / "citywide_sample.zip"

print("Creating zip (this may take a few minutes)...")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_STORED) as zf:  # STORED = no compression (videos don't compress well)
    for f in sorted(SAMPLE_DIR.rglob("*")):
        if f.is_file():
            arcname = str(f.relative_to(SAMPLE_DIR))
            zf.write(f, arcname)

print(f"\nDone: {ZIP_PATH.name} ({ZIP_PATH.stat().st_size/(1024**3):.2f} GB)")
print("Download from the Output tab on the right.")
print("\nExtract to data/raw/ in your local project:")
print("  data/raw/cityflowv2/S01_c001/vdo.avi ...")
print("  data/raw/wildtrack/videos/C1.mp4 ...")
print("  data/raw/wildtrack/annotations_positions/*.json")

In [ ]:
# Summary of what's in the sample
print("=" * 60)
print("SAMPLE CONTENTS")
print("=" * 60)

for ds in ["cityflowv2", "wildtrack"]:
    ds_dir = SAMPLE_DIR / ds
    if not ds_dir.exists(): continue
    print(f"\n{ds.upper()}:")
    for item in sorted(ds_dir.rglob("*")):
        if item.is_file() and item.suffix in (".avi", ".mp4", ".txt", ".json"):
            rel = item.relative_to(SAMPLE_DIR)
            size = item.stat().st_size / (1024**2)
            if size > 0.1:
                print(f"  {rel} ({size:.1f} MB)")
    # Count small files
    json_count = len(list(ds_dir.rglob("*.json")))
    if json_count > 10:
        print(f"  ... + {json_count} JSON annotation files")